# 🚇 Find My Nearest Delhi Metro Station

This notebook finds the Delhi Metro station closest to where you are **right now**.

**How it works, in 4 steps:**
1. Download the list of metro stations (name + location)
2. Get your live location from your browser
3. Measure the distance from you to every station
4. Show the closest ones on a map



## Install the libraries

We use just three libraries:
- `pandas`: works with tables of data
- `requests`: downloads data from the internet
- `folium`: draws interactive maps

In [ ]:
%pip install pandas requests folium

## Import the libraries

In [ ]:
import os
import math
import json
import time
import webbrowser
import http.server

import pandas as pd
import requests
import folium

print("Libraries ready ✅")

## Download the metro stations

We get station data for free from **OpenStreetMap**, a map of the world made by volunteers.

We ask it for every metro station inside a box drawn around Delhi NCR. The box is described by 4 numbers: its south, west, north and east edges.

The data is saved to `metro_stations.csv`, so next time it loads from your computer instead of downloading again.

In [ ]:
FILE_NAME = "metro_stations.csv"

# Three free OpenStreetMap servers. If one is busy, we try the next.
SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]

# A box around Delhi NCR: south, west, north, east
QUERY = """
[out:json][timeout:90];
nwr["railway"="station"]["station"="subway"](28.30,76.85,28.90,77.60);
out center;
"""

# The servers ask every program to say who it is
HEADERS = {"User-Agent": "delhi-metro-student-project/1.0"}


def download_stations():
    for attempt in range(2):                 # go through the servers twice
        for url in SERVERS:
            print("Trying", url)
            try:
                response = requests.post(url, data={"data": QUERY},
                                         headers=HEADERS, timeout=120)
                if response.status_code != 200:
                    print("   Server busy (code", response.status_code, "), trying next...")
                    continue
                data = response.json()
            except Exception as error:
                print("   Failed:", error)
                continue

            rows = []
            for place in data["elements"]:
                tags = place.get("tags", {})
                name = tags.get("name:en") or tags.get("name")
                lat = place.get("lat") or place.get("center", {}).get("lat")
                lon = place.get("lon") or place.get("center", {}).get("lon")
                if name and lat and lon:
                    rows.append({"name": name, "lat": lat, "lon": lon})

            if rows:
                stations = pd.DataFrame(rows).drop_duplicates(subset="name")
                stations.to_csv(FILE_NAME, index=False)
                print("✅ Downloaded and saved!")
                return stations

        print("All servers busy. Waiting 20 seconds and trying again...")
        time.sleep(20)

    raise RuntimeError("Could not download stations. Check your internet and try again in a few minutes.")


if os.path.exists(FILE_NAME):
    stations = pd.read_csv(FILE_NAME)
    print("Loaded stations from file")
else:
    print("Downloading stations... (takes a few seconds)")
    stations = download_stations()

print("Number of stations:", len(stations))
stations.head()


## Get your live location 📍

When you run the next cell:
1. A new tab opens in your browser
2. Your browser asks: **"Allow this site to know your location?"** Click **Allow**
3. Your coordinates are sent back here automatically

**How does this work?** Python starts a tiny website that runs only on your own computer (`localhost`). The page uses your browser's location feature (GPS on phones, Wi-Fi on laptops) and sends the result back to Python. Nothing leaves your computer.

You don't need to understand every line of this cell yet. Just know that `get_live_location()` gives you back your latitude and longitude.

In [ ]:
# The web page that asks the browser for your location
PAGE = '''
<html><body style="font-family:sans-serif;text-align:center;margin-top:80px">
<h2 id="msg">Getting your location... please click "Allow" 📍</h2>
<script>
function send(data) {
  fetch("/", {method: "POST", body: JSON.stringify(data)});
}
navigator.geolocation.getCurrentPosition(
  function (pos) {
    send({lat: pos.coords.latitude, lon: pos.coords.longitude, accuracy: pos.coords.accuracy});
    document.getElementById("msg").innerText = "✅ Location sent to Python! You can close this tab.";
  },
  function (err) {
    send({error: err.message});
    document.getElementById("msg").innerText = "❌ " + err.message;
  },
  {enableHighAccuracy: true, timeout: 20000}
);
</script></body></html>
'''


def get_live_location(wait_seconds=60):
    result = {}

    class Handler(http.server.BaseHTTPRequestHandler):
        def do_GET(self):            # the browser opens the page
            self.send_response(200)
            self.send_header("Content-Type", "text/html; charset=utf-8")
            self.end_headers()
            self.wfile.write(PAGE.encode("utf-8"))

        def do_POST(self):           # the page sends us the location
            length = int(self.headers["Content-Length"])
            result.update(json.loads(self.rfile.read(length)))
            self.send_response(200)
            self.end_headers()

        def log_message(self, *args):   # keep the output quiet
            pass

    server = http.server.HTTPServer(("localhost", 8765), Handler)
    server.timeout = 1
    webbrowser.open("http://localhost:8765")
    print("A browser tab has opened. Please click 'Allow'...")

    start = time.time()
    while not result and time.time() - start < wait_seconds:
        server.handle_request()
    server.server_close()

    if not result:
        raise TimeoutError("No location received. Did you click 'Allow'?")
    if "error" in result:
        raise RuntimeError("Browser said: " + result["error"])

    print(f"Accuracy: about {round(result['accuracy'])} metres")
    return result["lat"], result["lon"]


my_lat, my_lon = get_live_location()
print("Your location:", my_lat, my_lon)

**Didn't work?** You can type a location by hand instead. Remove the `#` from the line below and run it. (These coordinates are Connaught Place.)

In [ ]:
# my_lat, my_lon = 28.6315, 77.2167

## Measure distances

The Earth is round, so we can't use a simple ruler formula. We use the **Haversine formula**, which calculates the distance between two points on a sphere from their latitude and longitude.

In [ ]:
def distance_km(lat1, lon1, lat2, lon2):
    R = 6371  # radius of the Earth in km

    # convert degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


# Quick test: Connaught Place to India Gate (should be roughly 2.5 km)
print(round(distance_km(28.6315, 77.2167, 28.6129, 77.2295), 2), "km")

## Find the nearest stations

We work out the distance from you to **every** station, add it as a new column, then sort from closest to furthest.

Walking time is a rough guess: people walk about 5 km per hour, so 1 km takes about 12 minutes.

In [ ]:
stations["distance_km"] = stations.apply(
    lambda row: distance_km(my_lat, my_lon, row["lat"], row["lon"]),
    axis=1
)

nearest = stations.sort_values("distance_km").head(3).copy()
nearest["walk_minutes"] = (nearest["distance_km"] * 12).round()

closest = nearest.iloc[0]
print(f"🚇 Your nearest metro station is {closest['name']}")
print(f"   It is {closest['distance_km']:.2f} km away (about {closest['walk_minutes']:.0f} minutes walk)")

nearest

## Show it on a map 🗺️

- 🔴 Red marker: you
- 🟢 Green marker: the nearest station
- 🔵 Blue markers: the next closest stations

You can zoom and drag the map, and click a marker to see its name.

In [ ]:
my_map = folium.Map(location=[my_lat, my_lon], zoom_start=15)

# You
folium.Marker(
    [my_lat, my_lon],
    popup="You are here",
    icon=folium.Icon(color="red")
).add_to(my_map)

# The nearest stations
for i, (_, row) in enumerate(nearest.iterrows()):
    color = "green" if i == 0 else "blue"
    folium.Marker(
        [row["lat"], row["lon"]],
        popup=f"{row['name']} ({row['distance_km']:.2f} km)",
        icon=folium.Icon(color=color)
    ).add_to(my_map)

# A line from you to the closest station
folium.PolyLine(
    [[my_lat, my_lon], [closest["lat"], closest["lon"]]],
    color="green"
).add_to(my_map)

my_map

my_map.save("my_map.html")
webbrowser.open("file://" + os.path.abspath("my_map.html"))